# RAG Near-Zero Hallucination — Learning Edition

This is a **lightweight, learning-friendly** version of the full RAG notebook. It keeps the
same architecture and data flow (retrieve → constrain → verify → abstain) but swaps the
large H100-only models for small, permissively-licensed models that run comfortably on a
laptop CPU or small GPU.

**What this notebook is for.** Understanding the pipeline, experimenting with thresholds,
and seeing how the four hallucination-control layers work together. **It is NOT for
reproducing the full H100 numbers** or the 10M-vector scale-lab measurements.

**The four control layers (same as the full version):**

1. **Retrieve the right evidence** — hybrid (dense + BM25) + reranking.
2. **Constrain generation** — answer *only* from context, cite passage IDs, or abstain.
3. **Verify every atomic claim** against the cited context with an NLI faithfulness model.
4. **Abstain** when claim-support or retrieval-confidence falls below a calibrated threshold.


## Prerequisites & expected runtime

- **Python:** 3.10+
- **RAM:** ~8 GB (the tiny corpus + small models fit here)
- **GPU:** Optional — an Apple Silicon Mac, an integrated GPU, or a small discrete GPU will
  speed up generation, but the notebook also runs on CPU.
- **Disk:** a few GB for model downloads and checkpoints.
- **Expected run time:** 10–30 minutes on CPU for the default tiny config
  (`SLICE_SIZE=500`, 20+20 evaluation questions).

The first run downloads the models; subsequent runs reuse the Hugging Face cache.


## 0 · Setup, device check & global config

**Theory.** On a laptop, the hard budget is RAM and wall-clock time, not H100 VRAM. We load
a single small generative model directly into this kernel (no separate vLLM server), and we
use tiny CPU-friendly embedding/reranking/NLI models. Every config knob lives in one
`parameters`-tagged cell so you can override it easily; `LEARNING_MODE` guards the small-model
choices throughout the notebook.


In [1]:
# ---- papermill parameters (overridable from the CLI) -----------------------
LEARNING_MODE = True          # guard flag used throughout
SLICE_SIZE = 500              # tiny corpus
N_EVAL_ANSWERABLE = 20
N_EVAL_UNANSWERABLE = 20

# Small, permissively-licensed, CPU-friendly models
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"                 # or fallback "microsoft/Phi-3-mini-4k-instruct"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
USE_LLM_JUDGE = False                         # NLI verifier only (no huge LLM judge)

CHUNK_TOKENS = 128
CHUNK_OVERLAP = 16
RETRIEVE_K = 50
RERANK_TOP_N = 5
RRF_K = 60
MAX_HOPS = 2
CRAG_OK = 0.65
CRAG_BAD = 0.35
TAU_CLAIM = 0.50
TAU_ABSTAIN = 0.50

# Local, project-relative paths (works on macOS and Linux)
DATA_DIR = "./learning_data"
SCRATCH_DIR = "./learning_scratch"
SEED = 42
RUN_SCALE_LAB = False
RUN_CONTEXTUALIZE = False     # skip expensive per-chunk LLM contextualization for learning


In [2]:
# ---- resolve learning profile and local paths --------------------------------
import os
from pathlib import Path

if not LEARNING_MODE:
    raise RuntimeError("This notebook is the learning edition; keep LEARNING_MODE=True")

# Force the small-model choices no matter what was injected
RUN_SCALE_LAB = False
RUN_CONTEXTUALIZE = False
USE_LLM_JUDGE = False

DATA_DIR = Path(DATA_DIR).resolve()
SCRATCH_DIR = Path(SCRATCH_DIR).resolve()
ART_DIR = DATA_DIR / "artifacts"     # checkpoints: chunks, embeddings, indexes
OUT_DIR = DATA_DIR / "out"
FIG_DIR = OUT_DIR / "figures"
for d in (ART_DIR, OUT_DIR, FIG_DIR, DATA_DIR / "datasets", DATA_DIR / "hf"):
    Path(d).mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(DATA_DIR / "hf"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print(f"LEARNING  slice={SLICE_SIZE}  eval={N_EVAL_ANSWERABLE}+{N_EVAL_UNANSWERABLE}  "
      f"artifacts={ART_DIR}")


LEARNING  slice=500  eval=20+20  artifacts=/Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/artifacts


In [3]:
# ---- imports + determinism -------------------------------------------------
import json
import random
import subprocess
import time
from dataclasses import dataclass, asdict, field

import numpy as np


def set_determinism(seed: int) -> None:
    """Seed every RNG we touch so runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    except Exception:
        pass


set_determinism(SEED)


In [4]:
# ---- CPU / GPU info helpers ------------------------------------------------
import torch


def device_report() -> dict:
    """Print CPU/GPU info. Do NOT assert a specific GPU — this notebook must run on laptops."""
    rep = {"platform": "CPU"}
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        rep = {"platform": "CUDA", "device": name, "total_gb": round(total, 1)}
    elif torch.backends.mps.is_available():
        rep = {"platform": "MPS", "device": "Apple Silicon / Metal"}
    print(json.dumps(rep, indent=2))
    return rep


def memory_snapshot(tag: str) -> dict:
    """Log kernel-side memory after a load step."""
    snap = {"tag": tag}
    if torch.cuda.is_available():
        snap["cuda_allocated_gb"] = round(torch.cuda.memory_allocated() / 1024**3, 2)
        snap["cuda_reserved_gb"] = round(torch.cuda.memory_reserved() / 1024**3, 2)
        print(f"[mem] {tag:22} cuda_allocated={snap['cuda_allocated_gb']}GB  "
              f"reserved={snap['cuda_reserved_gb']}GB")
    else:
        print(f"[mem] {tag:22} running on CPU")
    return snap


GPU = device_report()   # kept as GPU for compatibility with downstream manifest code


{
  "platform": "MPS",
  "device": "Apple Silicon / Metal"
}


In [5]:
# ---- frozen run config -----------------------------------------------------
@dataclass
class Config:
    learning: bool
    slice_size: int
    gen_model: str
    embed_model: str
    rerank_model: str
    nli_model: str
    use_llm_judge: bool
    chunk_tokens: int
    chunk_overlap: int
    retrieve_k: int
    rerank_top_n: int
    rrf_k: int
    max_hops: int
    crag_ok: float
    crag_bad: float
    tau_claim: float
    tau_abstain: float
    seed: int

    def summary(self) -> dict:
        return asdict(self)


CFG = Config(
    learning=LEARNING_MODE, slice_size=SLICE_SIZE, gen_model=GEN_MODEL,
    embed_model=EMBED_MODEL, rerank_model=RERANK_MODEL, nli_model=NLI_MODEL,
    use_llm_judge=USE_LLM_JUDGE, chunk_tokens=CHUNK_TOKENS, chunk_overlap=CHUNK_OVERLAP,
    retrieve_k=RETRIEVE_K, rerank_top_n=RERANK_TOP_N, rrf_k=RRF_K, max_hops=MAX_HOPS,
    crag_ok=CRAG_OK, crag_bad=CRAG_BAD, tau_claim=TAU_CLAIM, tau_abstain=TAU_ABSTAIN,
    seed=SEED,
)
print(json.dumps(CFG.summary(), indent=2))


{
  "learning": true,
  "slice_size": 500,
  "gen_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
  "rerank_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "nli_model": "cross-encoder/nli-deberta-v3-base",
  "use_llm_judge": false,
  "chunk_tokens": 128,
  "chunk_overlap": 16,
  "retrieve_k": 50,
  "rerank_top_n": 5,
  "rrf_k": 60,
  "max_hops": 2,
  "crag_ok": 0.65,
  "crag_bad": 0.35,
  "tau_claim": 0.5,
  "tau_abstain": 0.5,
  "seed": 42
}


In [6]:
# ---- local generative LLM via transformers ---------------------------------
# For learning we load a single small model directly in this kernel instead of talking to a
# separate vLLM server. This avoids the H100 server setup and is portable to CPU / MPS / small GPU.
from transformers import AutoModelForCausalLM, AutoTokenizer


class LocalLLM:
    """Small local causal-LM loaded with transformers."""

    def __init__(self, model_name: str, device: str | None = None):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        load_kwargs = {"trust_remote_code": True}
        if torch.cuda.is_available():
            # device_map="auto" places layers across available GPUs / unified memory.
            load_kwargs.update({"torch_dtype": torch.float16, "device_map": "auto"})
        else:
            load_kwargs["torch_dtype"] = torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        self.model.eval()
        print(f"[llm] loaded {model_name} on {self.model.device}")

    def is_up(self) -> bool:
        return self.model is not None

    def chat(self, system: str, user: str, temperature: float = 0.0,
             max_tokens: int = 512, stop: list[str] | None = None) -> str:
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": user}]
        if hasattr(self.tokenizer, "apply_chat_template"):
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"{system}\n\n{user}"
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature if temperature > 0 else None,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        generated = outputs[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)
        if stop:
            for s in stop:
                if s in text:
                    text = text[:text.index(s)]
        return text.strip()

    def batch_chat(self, system: str, users: list[str], **kw) -> list[str]:
        return [self.chat(system, u, **kw) for u in users]


llm = LocalLLM(GEN_MODEL)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[llm] loaded Qwen/Qwen2.5-1.5B-Instruct on cpu


## 1 · Download datasets & build the corpus slice

**Theory.** Good evaluation needs four strata, and our datasets are chosen to provide
all of them:
- **HotpotQA (distractor)** — multi-hop questions *with sentence-level gold supporting
  facts*. Those gold labels let us measure **context-recall** honestly, and its bundled
  Wikipedia paragraphs *are* our corpus.
- **FRAMES** — held-out multi-hop factuality (kept separate to avoid judge leakage).
- **SQuAD v2 (`is_impossible`)** + **Confabulations** + hand-built **false-premise** Qs —
  the **unanswerable** stratum that proves the system *abstains* instead of bluffing.
- **HaluBench** — used in Sec 14 to validate the verifier itself.

We slice HotpotQA to `SLICE_SIZE` passages by unioning each question's context
paragraphs (gold + distractors), deduping by title, and keeping provenance so we can
score retrieval against the gold titles later.

In [7]:
from datasets import load_dataset

DS_CACHE = str(DATA_DIR / "datasets")


def load_hotpotqa(split: str = "validation"):
    # datasets 3.x requires the namespaced repo id 'hotpotqa/hotpot_qa'
    return load_dataset("hotpotqa/hotpot_qa", "distractor", split=split, cache_dir=DS_CACHE)


def load_squad_v2(split: str = "validation"):
    return load_dataset("rajpurkar/squad_v2", split=split, cache_dir=DS_CACHE)


def load_frames():
    try:
        return load_dataset("google/frames-benchmark", split="test", cache_dir=DS_CACHE)
    except Exception as e:
        print(f"[frames] skipped ({e})"); return None


def load_halubench():
    try:
        return load_dataset("PatronusAI/HaluBench", split="test", cache_dir=DS_CACHE)
    except Exception as e:
        print(f"[halubench] skipped ({e})"); return None


hotpot = load_hotpotqa()
print(f"[data] hotpotqa(validation) = {len(hotpot)} questions")

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

[data] hotpotqa(validation) = 7405 questions


In [8]:
import hashlib
from dataclasses import dataclass


@dataclass
class Passage:
    id: str
    title: str
    text: str
    is_gold_for: list[str] = field(default_factory=list)  # question ids this is gold evidence for


@dataclass
class QAItem:
    qid: str
    question: str
    answer: str
    answerable: bool
    gold_titles: list[str] = field(default_factory=list)
    gold_sentences: list[str] = field(default_factory=list)
    source: str = "hotpotqa"
    qtype: str = ""   # bridge | comparison | unanswerable | false_premise


def _pid(title: str, idx: int) -> str:
    return hashlib.md5(f"{title}::{idx}".encode()).hexdigest()[:12]


class CorpusBuilder:
    """Build a passage corpus + QA items from HotpotQA distractor contexts."""

    def build(self, qa, n_passages: int):
        passages: dict[str, Passage] = {}
        qa_items: list[QAItem] = []
        for ex in qa:
            qid = ex["id"]
            gold_titles = list(dict.fromkeys(ex["supporting_facts"]["title"]))
            gold_sent_ids = ex["supporting_facts"]["sent_id"]
            titles = ex["context"]["title"]
            sents = ex["context"]["sentences"]
            # gold supporting sentences (text) for this question
            gold_sentences = []
            title_to_sents = {t: s for t, s in zip(titles, sents)}
            for gt, sid in zip(ex["supporting_facts"]["title"], gold_sent_ids):
                ss = title_to_sents.get(gt, [])
                if 0 <= sid < len(ss):
                    gold_sentences.append(ss[sid].strip())
            # add every context paragraph as a passage
            for t, ss in zip(titles, sents):
                para = " ".join(s.strip() for s in ss).strip()
                if len(para) < 40:
                    continue
                pid = _pid(t, 0)
                if pid not in passages:
                    passages[pid] = Passage(id=pid, title=t, text=para)
                if t in gold_titles and qid not in passages[pid].is_gold_for:
                    passages[pid].is_gold_for.append(qid)
            qa_items.append(QAItem(
                qid=qid, question=ex["question"], answer=ex["answer"], answerable=True,
                gold_titles=gold_titles, gold_sentences=gold_sentences,
                source="hotpotqa", qtype=ex.get("type", ""),
            ))
            if len(passages) >= n_passages:
                break
        return list(passages.values()), qa_items


corpus, qa_items = CorpusBuilder().build(hotpot, SLICE_SIZE)
print(f"[corpus] passages={len(corpus)}  qa_items={len(qa_items)}  "
      f"gold-bearing passages={sum(1 for p in corpus if p.is_gold_for)}")

[corpus] passages=501  qa_items=51  gold-bearing passages=102


In [9]:
# ---- quick visibility: sizes + a sample question with its gold evidence -----
import pandas as pd

tok_lens = [len(p.text.split()) for p in corpus]
print(pd.Series(tok_lens, name="passage_word_count").describe().round(1).to_string())

_ex = qa_items[0]
print("\nSample question:")
print(f"  Q: {_ex.question}")
print(f"  A: {_ex.answer}   (type={_ex.qtype})")
print(f"  gold titles: {_ex.gold_titles}")
for s in _ex.gold_sentences:
    print(f"   - {s}")

count     501.0
mean       97.8
std        79.6
min        12.0
25%        58.0
50%        84.0
75%       114.0
max      1309.0

Sample question:
  Q: Were Scott Derrickson and Ed Wood of the same nationality?
  A: yes   (type=comparison)
  gold titles: ['Scott Derrickson', 'Ed Wood']
   - Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.
   - Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.


## 2 · Parse / normalize / dedup

**Theory.** Garbage in, hallucinations out. Two cheap cleanups pay off disproportionately:
(1) **normalization** (unicode NFKC, whitespace) so BM25 tokenization is stable, and
(2) **near-duplicate removal** — copied/forwarded passages otherwise crowd the top-k and
inflate "retrieval" without adding evidence. We use **MinHash LSH** (Jaccard over token
shingles) to drop near-dups in roughly linear time.

In [10]:
import re
import unicodedata


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("­", "")            # soft hyphen
    s = re.sub(r"[ \t ]+", " ", s)    # collapse spaces
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

In [11]:
from datasketch import MinHash, MinHashLSH


class Deduper:
    """Drop near-duplicate passages via MinHash LSH over word shingles."""

    def __init__(self, threshold: float = 0.9, num_perm: int = 64):
        self.threshold = threshold
        self.num_perm = num_perm

    def _mh(self, text: str) -> MinHash:
        m = MinHash(num_perm=self.num_perm)
        for tok in set(text.lower().split()):
            m.update(tok.encode())
        return m

    def fit_transform(self, passages: list[Passage]):
        lsh = MinHashLSH(threshold=self.threshold, num_perm=self.num_perm)
        kept, dropped = [], 0
        for p in passages:
            m = self._mh(p.text)
            if lsh.query(m):          # a near-duplicate already kept
                dropped += 1
                continue
            lsh.insert(p.id, m)
            kept.append(p)
        return kept, {"kept": len(kept), "dropped_near_dup": dropped}

In [12]:
def quality_filter(p: Passage, min_words: int = 8) -> bool:
    return len(p.text.split()) >= min_words and any(c.isalpha() for c in p.text)


def run_clean(passages: list[Passage]):
    n0 = len(passages)
    for p in passages:
        p.text = normalize_text(p.text)
    filtered = [p for p in passages if quality_filter(p)]
    deduped, stats = Deduper(threshold=0.9).fit_transform(filtered)
    stats.update({"input": n0, "after_quality": len(filtered), "after_dedup": len(deduped)})
    return deduped, stats


corpus, clean_stats = run_clean(corpus)
print(json.dumps(clean_stats, indent=2))

{
  "kept": 501,
  "dropped_near_dup": 0,
  "input": 501,
  "after_quality": 501,
  "after_dedup": 501
}


## 3 · Structure-aware chunking

**Theory.** Fixed-size chunking severs the entity-bearing sentence from the context
that disambiguates it — fatal for multi-hop questions. We pack **whole sentences** up to
a token budget (`CHUNK_TOKENS`) with a small **overlap**, using the *generator's own
tokenizer* so token counts match what the LLM will actually see. Each chunk keeps a
stable `id` so we can cite it precisely later.

**Learning note.** We use `CHUNK_TOKENS=128` (half the full notebook) so the small model's
context window is never stressed and indexing stays fast.


In [13]:
# ---- structure-aware chunker using the small generator tokenizer ------------
from transformers import AutoTokenizer

_SENT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list[str]:
    return [s.strip() for s in _SENT_RE.split(text) if s.strip()]


@dataclass
class Chunk:
    id: str
    passage_id: str
    title: str
    text: str
    token_count: int
    contextual_text: str = ""   # filled in Section 4

    @property
    def index_text(self) -> str:
        return self.contextual_text or self.text


class StructureAwareChunker:
    def __init__(self, tokenizer, target_tokens: int = 128, overlap: int = 16):
        self.tok = tokenizer
        self.target = target_tokens
        self.overlap = overlap

    def _ntok(self, text: str) -> int:
        return len(self.tok.encode(text, add_special_tokens=False))

    def _make(self, passage: Passage, sents: list[str], n: int) -> Chunk:
        text = " ".join(sents)
        cid = _pid(passage.title, hash(text) & 0xFFFFFF)
        return Chunk(id=cid, passage_id=passage.id, title=passage.title,
                     text=text, token_count=self._ntok(text))

    def chunk(self, passage: Passage) -> list[Chunk]:
        sents = split_sentences(passage.text) or [passage.text]
        chunks, cur, cur_tok, n = [], [], 0, 0
        for s in sents:
            st = self._ntok(s)
            if cur and cur_tok + st > self.target:
                chunks.append(self._make(passage, cur, n)); n += 1
                # overlap: carry the trailing sentence into the next chunk
                cur, cur_tok = ([cur[-1]], self._ntok(cur[-1])) if self.overlap else ([], 0)
            cur.append(s); cur_tok += st
        if cur:
            chunks.append(self._make(passage, cur, n))
        return chunks


def chunk_corpus(passages: list[Passage], chunker: StructureAwareChunker) -> list[Chunk]:
    out: list[Chunk] = []
    for p in passages:
        out.extend(chunker.chunk(p))
    return out


_tok = AutoTokenizer.from_pretrained(GEN_MODEL, trust_remote_code=True)
chunker = StructureAwareChunker(_tok, CHUNK_TOKENS, CHUNK_OVERLAP)
chunks = chunk_corpus(corpus, chunker)
_ct = [c.token_count for c in chunks]
print(f"[chunk] {len(corpus)} passages -> {len(chunks)} chunks  "
      f"(tokens: mean={np.mean(_ct):.0f} p95={np.percentile(_ct, 95):.0f})")


[chunk] 501 passages -> 894 chunks  (tokens: mean=95 p95=128)


## 4 · Contextual Retrieval (optional, gated off for learning)

**Theory.** A chunk like *"revenue grew 3% that quarter"* is unsearchable in isolation —
*whose* revenue, *which* quarter? Anthropic's **Contextual Retrieval** prepends a one-line,
LLM-written *situating context* to each chunk before indexing, which can lift recall.

**Learning note.** Contextualization calls the LLM once per chunk, so it is the most
expensive offline step. We gate it behind `RUN_CONTEXTUALIZE`. In the learning run it is
`False`, so chunks keep their raw text and the pipeline stays fast. To try it, set
`RUN_CONTEXTUALIZE=True` and re-run this section.


In [14]:
# ---- optional contextualization (skipped by default in learning mode) --------
import pickle

CONTEXTUALIZE_PROMPT = (
    "Here is a document titled '{title}':\n<document>\n{doc}\n</document>\n\n"
    "Here is a chunk from it:\n<chunk>\n{chunk}\n</chunk>\n\n"
    "Give a short, single-sentence context (<=25 words) that situates this chunk within "
    "the document so it can be retrieved on its own. Answer with the sentence only."
)


class Contextualizer:
    def __init__(self, llm: LocalLLM):
        self.llm = llm

    @staticmethod
    def estimate(n_chunks: int, sec_per_call: float = 1.5) -> dict:
        return {"chunks": n_chunks, "approx_minutes": round(n_chunks * sec_per_call / 60, 1)}

    def contextualize(self, chunks: list[Chunk], doc_lookup: dict[str, str]) -> list[Chunk]:
        print(f"[ctx] estimate: {self.estimate(len(chunks))} (one LLM call per chunk)")
        sys_prompt = "You write concise retrieval context. Output one sentence, nothing else."
        for c in chunks:
            doc = doc_lookup.get(c.passage_id, c.text)[:2000]
            user = CONTEXTUALIZE_PROMPT.format(title=c.title, doc=doc, chunk=c.text)
            try:
                ctx = self.llm.chat(sys_prompt, user, max_tokens=64).strip().replace("\n", " ")
            except Exception:
                ctx = ""
            c.contextual_text = (ctx + "\n" + c.text) if ctx else c.text
        return chunks


def contextualize_or_load(chunks: list[Chunk]) -> list[Chunk]:
    ckpt = ART_DIR / f"chunks_ctx_{len(chunks)}.pkl"
    if ckpt.exists():
        print(f"[ctx] loading checkpoint {ckpt.name}")
        return pickle.load(open(ckpt, "rb"))
    if not RUN_CONTEXTUALIZE:
        print("[ctx] RUN_CONTEXTUALIZE=False; keeping raw chunk text (fast learning path).")
        for c in chunks:
            c.contextual_text = c.text
        return chunks
    doc_lookup = {p.id: p.text for p in corpus}
    out = Contextualizer(llm).contextualize(chunks, doc_lookup)
    pickle.dump(out, open(ckpt, "wb"))
    return out


_t0 = time.time()
chunks = contextualize_or_load(chunks)
print(f"[ctx] done in {time.time() - _t0:.1f}s")
_demo = next((c for c in chunks if c.contextual_text != c.text), chunks[0])
print("\nBefore:\n ", _demo.text[:200])
print("After (context-prefixed):\n ", _demo.contextual_text[:260])


[ctx] RUN_CONTEXTUALIZE=False; keeping raw chunk text (fast learning path).
[ctx] done in 0.0s

Before:
  Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film concerns the period in Wood's life wh
After (context-prefixed):
  Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film concerns the period in Wood's life when he made his best-known films as well as his relationship 


## 5 · Load the local auxiliary models

**Theory.** The small generator is already loaded. Here we load the three *kernel-side*
models: the **reranker** (a tiny cross-encoder), the **faithfulness verifier** (an NLI
cross-encoder), and — in Section 6 — the **embedder**. All are small enough for a laptop;
they use CPU by default and will use CUDA/MPS automatically if available.


In [15]:
# ---- small cross-encoder reranker -------------------------------------------
from sentence_transformers import CrossEncoder


class CrossEncoderReranker:
    """Score (query, doc) pairs with a cross-encoder. Higher = more relevant."""

    def __init__(self, name: str):
        device = "cuda" if torch.cuda.is_available() else None   # None -> CPU / MPS auto
        self.model = CrossEncoder(name, device=device)

    def score(self, query: str, docs: list[str], batch_size: int = 8) -> list[float]:
        if not docs:
            return []
        pairs = [(query, d) for d in docs]
        scores = self.model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        # ms-marco-MiniLM returns a single score per pair
        import numpy as np
        scores = np.atleast_1d(scores)
        return [float(s) for s in scores]


reranker = CrossEncoderReranker(RERANK_MODEL)
memory_snapshot("reranker")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[mem] reranker               running on CPU


{'tag': 'reranker'}

In [16]:
# ---- NLI faithfulness verifier (no MiniCheck, no LLM judge) -----------------
class Verifier:
    """Faithfulness via a natural-language-inference cross-encoder: does the `context`
    ENTAIL the `claim`? We intentionally use ONLY the small NLI model in learning mode —
    no huge MiniCheck model and no LLM-as-judge, so the notebook stays laptop-friendly."""

    def __init__(self, nli_name: str):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(nli_name)
        load_kwargs = {}
        if torch.cuda.is_available():
            load_kwargs.update({"torch_dtype": torch.float16, "device_map": "auto"})
        self.model = AutoModelForSequenceClassification.from_pretrained(nli_name, **load_kwargs)
        if not torch.cuda.is_available():
            self.model = self.model.to(self.device)
        self.model.eval()
        id2label = {int(i): str(l).lower() for i, l in self.model.config.id2label.items()}
        self.entail_idx = next((i for i, l in id2label.items() if "entail" in l), 1)

    @torch.no_grad()
    def nli_score(self, claim: str, context: str) -> float:
        # P(context entails claim); premise=context, hypothesis=claim
        enc = self.tok(context, claim, return_tensors="pt", truncation=True,
                       max_length=512).to(self.model.device)
        probs = torch.softmax(self.model(**enc).logits[0].float(), dim=-1)
        return float(probs[self.entail_idx])

    def support(self, claim: str, context: str) -> dict:
        n = self.nli_score(claim, context)
        return {"score": n, "nli": n, "minicheck": None, "agree": None}


# Hard-guard: never use the LLM judge in learning mode, never load MiniCheck.
USE_LLM_JUDGE = False
verifier = Verifier(NLI_MODEL)
memory_snapshot("verifier")


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[mem] verifier               running on CPU


{'tag': 'verifier'}

In [17]:
# ---- memory sanity check (laptop-friendly) ---------------------------------
def assert_memory_budget(limit_gb: float = 10.0) -> None:
    """Warn if a single kernel-side model looks unexpectedly large; do not assert H100 VRAM."""
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        print(f"[mem] models allocated ~{used:.1f}GB CUDA so far (limit check={limit_gb}GB)")
        assert used <= limit_gb, f"CUDA memory over learning budget: {used:.1f}GB"
    else:
        print("[mem] running on CPU; no VRAM budget assertion.")


assert_memory_budget()


[mem] running on CPU; no VRAM budget assertion.


## 6 · Build the hybrid index (LanceDB dense + bm25s sparse)

**Theory.** Dense embeddings catch paraphrase; BM25 catches exact tokens (names, IDs,
numbers) that dense models blur. We index both, keyed by `chunk.id`, over the
**contextualized** chunk text. LanceDB is an embedded, on-disk (NVMe) vector store — no
server — and bm25s is a fast sparse index. We checkpoint both so reruns skip the rebuild.

In [18]:
# ---- small sentence-transformer embedder (CPU/GPU auto) --------------------
def load_embedder(name: str):
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else None   # None lets ST pick CPU/MPS
    return SentenceTransformer(name, device=device)


def embed_texts(embedder, texts: list[str], batch_size: int = 16,
                is_query: bool = False) -> np.ndarray:
    # all-MiniLM-L6-v2 does not use instruction prompts; we only normalize embeddings.
    return embedder.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False).astype("float32")


In [19]:
import lancedb


class LanceVectorStore:
    def __init__(self, uri: str, table: str = "chunks"):
        self.db = lancedb.connect(uri)
        self.table_name = table
        self.tbl = None

    def build(self, chunks: list[Chunk], vectors: np.ndarray):
        rows = [{"id": c.id, "vector": vectors[i], "title": c.title, "text": c.text}
                for i, c in enumerate(chunks)]
        self.tbl = self.db.create_table(self.table_name, data=rows, mode="overwrite")
        return self

    def open(self):
        self.tbl = self.db.open_table(self.table_name)
        return self

    def search(self, qvec: np.ndarray, k: int) -> list[tuple[str, float]]:
        res = self.tbl.search(qvec).metric("cosine").limit(k).to_list()
        # cosine "_distance" in [0,2]; convert to similarity in [0,1]
        return [(r["id"], 1.0 - r["_distance"] / 2.0) for r in res]

In [20]:
import bm25s
import Stemmer


class BM25Index:
    def __init__(self):
        self.stemmer = Stemmer.Stemmer("english")
        self.retriever = None
        self.ids: list[str] = []

    def __getstate__(self):                 # PyStemmer (Cython) is not picklable
        d = self.__dict__.copy()
        d["stemmer"] = None
        return d

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.stemmer = Stemmer.Stemmer("english")

    def build(self, chunks: list[Chunk]):
        self.ids = [c.id for c in chunks]
        tokens = bm25s.tokenize([c.index_text for c in chunks], stemmer=self.stemmer)
        self.retriever = bm25s.BM25()
        self.retriever.index(tokens)
        return self

    def search(self, query: str, k: int) -> list[tuple[str, float]]:
        q = bm25s.tokenize(query, stemmer=self.stemmer)
        idx, scores = self.retriever.retrieve(q, k=min(k, len(self.ids)))
        return [(self.ids[int(i)], float(s)) for i, s in zip(idx[0], scores[0])]

In [21]:
# ---- build or load the hybrid index (single small embedder) -----------------
def build_or_load_index(chunks: list[Chunk]):
    lance_uri = str(ART_DIR / "lancedb")
    bm25_ckpt = ART_DIR / f"bm25_{len(chunks)}.pkl"
    vec = LanceVectorStore(lance_uri)
    bm25 = BM25Index()
    if (ART_DIR / "lancedb").exists() and bm25_ckpt.exists():
        print("[index] loading checkpoints")
        vec.open()
        bm25 = pickle.load(open(bm25_ckpt, "rb"))
        return vec, bm25
    emb = load_embedder(EMBED_MODEL)
    memory_snapshot("embedder")
    t0 = time.time()
    vectors = embed_texts(emb, [c.index_text for c in chunks])
    print(f"[index] embedded {len(chunks)} chunks in {time.time()-t0:.1f}s, dim={vectors.shape[1]}")
    vec.build(chunks, vectors)
    bm25.build(chunks)
    pickle.dump(bm25, open(bm25_ckpt, "wb"))
    return vec, bm25


vec_store, bm25_index = build_or_load_index(chunks)
print(f"[index] LanceDB on-disk: {ART_DIR / 'lancedb'}  |  bm25 over {len(bm25_index.ids)} chunks")

id_to_chunk = {c.id: c for c in chunks}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Split strings:   0%|          | 0/894 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/894 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/894 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/894 [00:00<?, ?it/s]

[index] LanceDB on-disk: /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/artifacts/lancedb  |  bm25 over 894 chunks


## 7 · Retrieval + Reciprocal Rank Fusion + reranking

**Theory.** Two-stage retrieval: cheap **recall** then expensive **precision**.
(1) Run dense + BM25, fuse with **Reciprocal Rank Fusion** — `score = Σ 1/(k+rank)` —
which is scale-free (no score normalization) and robust. (2) Take the top ~150 fused
candidates and **rerank** them with a cross-encoder to ~20. We then *prove* the lift by
measuring **passage-recall vs HotpotQA gold titles**: dense → hybrid → reranked.

In [22]:
# ---- load the online embedder (same small model; no model swap needed) --------
online_embedder = load_embedder(EMBED_MODEL)
memory_snapshot("embedder(online)")


@dataclass
class RetrievedChunk:
    id: str
    title: str
    text: str
    score: float
    source: str = "hybrid"


def rrf_fuse(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, cid in enumerate(ranking):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[mem] embedder(online)       running on CPU


In [23]:
class HybridRetriever:
    def __init__(self, vec_store, bm25, embedder, id_to_chunk, rrf_k=60):
        self.vec, self.bm25, self.embedder = vec_store, bm25, embedder
        self.id_to_chunk, self.rrf_k = id_to_chunk, rrf_k

    def _mk(self, cid: str, score: float, source: str) -> RetrievedChunk | None:
        c = self.id_to_chunk.get(cid)
        if not c:
            return None
        return RetrievedChunk(id=cid, title=c.title, text=c.text, score=score, source=source)

    def retrieve(self, query: str, k: int) -> list[RetrievedChunk]:
        qvec = embed_texts(self.embedder, [query], is_query=True)[0]
        dense = self.vec.search(qvec, k)
        sparse = self.bm25.search(query, k)
        fused = rrf_fuse([[i for i, _ in dense], [i for i, _ in sparse]], self.rrf_k)[:k]
        return [c for c in (self._mk(cid, s, "hybrid") for cid, s in fused) if c]

    def dense_only(self, query: str, k: int) -> list[RetrievedChunk]:
        qvec = embed_texts(self.embedder, [query], is_query=True)[0]
        return [c for c in (self._mk(cid, s, "dense") for cid, s in self.vec.search(qvec, k)) if c]


class RerankerStage:
    def __init__(self, reranker):
        self.reranker = reranker

    def rerank(self, query: str, cands: list[RetrievedChunk], top_n: int) -> list[RetrievedChunk]:
        if not cands:
            return []
        scores = self.reranker.score(query, [c.text for c in cands])
        ranked = sorted(zip(cands, scores), key=lambda x: -x[1])[:top_n]
        out = []
        for c, s in ranked:
            c.score, c.source = float(s), "reranked"
            out.append(c)
        return out


retriever = HybridRetriever(vec_store, bm25_index, online_embedder, id_to_chunk, RRF_K)
reranker_stage = RerankerStage(reranker)


def retrieve_and_rerank(query: str) -> list[RetrievedChunk]:
    return reranker_stage.rerank(query, retriever.retrieve(query, RETRIEVE_K), RERANK_TOP_N)

In [24]:
def passage_recall(retrieved: list[RetrievedChunk], gold_titles: list[str]) -> float:
    if not gold_titles:
        return float("nan")
    got = {r.title for r in retrieved}
    return len(got & set(gold_titles)) / len(set(gold_titles))


_q = qa_items[0]
_dense = retriever.dense_only(_q.question, RERANK_TOP_N)
_hybrid = retriever.retrieve(_q.question, RERANK_TOP_N)
_reranked = retrieve_and_rerank(_q.question)
print(f"Q: {_q.question}\n gold titles: {_q.gold_titles}")
print(f" recall@{RERANK_TOP_N}:  dense={passage_recall(_dense, _q.gold_titles):.2f}  "
      f"hybrid={passage_recall(_hybrid, _q.gold_titles):.2f}  "
      f"reranked={passage_recall(_reranked, _q.gold_titles):.2f}")
print(" top-3 reranked:")
for r in _reranked[:3]:
    print(f"   [{r.id}] ({r.score:.3f}) {r.title}: {r.text[:90]}...")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Q: Were Scott Derrickson and Ed Wood of the same nationality?
 gold titles: ['Scott Derrickson', 'Ed Wood']
 recall@5:  dense=0.50  hybrid=0.50  reranked=1.00
 top-3 reranked:
   [3826e24a4ed8] (3.394) Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. ...
   [c7078941edd9] (0.588) Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by ...
   [374121b82b28] (-1.593) Ed Wood: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, ac...


## 8 · Query routing & decomposition

**Theory.** Not every query needs the full agent. An **adaptive router** labels a query
`no_retrieval` (chit-chat / unanswerable-by-design), `single_hop`, or `multi_hop`, so we
spend compute only where it helps. For `multi_hop`, a **decomposer** splits the question
into ordered sub-questions we retrieve for separately — the basis of the agent loop. A
cheap **false-premise** detector lets us abstain early on presupposition-failure queries.

In [25]:
ROUTER_PROMPT = (
    "Classify the question into exactly one label:\n"
    "- no_retrieval: greetings/opinions or questions no document corpus could answer\n"
    "- single_hop: answerable by finding one fact\n"
    "- multi_hop: needs combining facts from multiple documents\n"
    "Question: {q}\nReply with only the label."
)


class QueryRouter:
    LABELS = {"no_retrieval", "single_hop", "multi_hop"}

    def __init__(self, llm: LocalLLM):
        self.llm = llm

    def route(self, query: str) -> str:
        out = self.llm.chat("You are a precise query classifier.",
                            ROUTER_PROMPT.format(q=query), max_tokens=8).strip().lower()
        for lbl in self.LABELS:
            if lbl in out:
                return lbl
        return "single_hop"

In [26]:
DECOMPOSE_PROMPT = (
    "Break this multi-hop question into 2-3 ordered, self-contained sub-questions, one per "
    "line, no numbering. If it is already simple, return it unchanged.\nQuestion: {q}"
)
FALSE_PREMISE_PROMPT = (
    "Does this question assume a fact that may be false or unverifiable? "
    "Answer yes or no.\nQuestion: {q}"
)


class QueryDecomposer:
    def __init__(self, llm: LocalLLM, max_hops: int = 3):
        self.llm, self.max_hops = llm, max_hops

    def decompose(self, query: str) -> list[str]:
        out = self.llm.chat("You decompose questions for retrieval.",
                            DECOMPOSE_PROMPT.format(q=query), max_tokens=160)
        subs = [ln.strip(" -•\t") for ln in out.splitlines() if ln.strip()]
        return (subs or [query])[: self.max_hops]


def detect_false_premise(query: str, llm: LocalLLM) -> bool:
    return llm.chat("You detect false presuppositions.",
                    FALSE_PREMISE_PROMPT.format(q=query), max_tokens=4).strip().lower().startswith("y")


router = QueryRouter(llm)
decomposer = QueryDecomposer(llm, MAX_HOPS)

if llm.is_up():
    _mh = next((q for q in qa_items if q.qtype == "comparison"), qa_items[0])
    print(f"route('{_mh.question[:60]}...') -> {router.route(_mh.question)}")
    print("decompose ->")
    for s in decomposer.decompose(_mh.question):
        print(f"   • {s}")
else:
    print("[route] model not loaded; skipping live demo")

route('Were Scott Derrickson and Ed Wood of the same nationality?...') -> no_retrieval
decompose ->
   • 1. Who was Scott Derrickson?
   • 2. Who was Ed Wood?


## 9 · Constrained, cited generation

**Theory.** This is **hallucination firewall #1**. The system prompt forbids outside
knowledge, requires an inline `[chunk_id]` citation for every claim, and provides an
explicit **abstain token** when the context is insufficient. We then *validate* citations
against the real chunk IDs and drop any the model invented — so a fabricated citation can
never survive to the user.


In [27]:
ABSTAIN_TOKEN = "INSUFFICIENT_EVIDENCE"
GENERATION_SYSTEM_PROMPT = (
    "You answer strictly from the numbered context passages. Rules:\n"
    "1. Use ONLY facts in the passages — never outside knowledge.\n"
    f"2. If the passages do not contain the answer, reply with exactly: {ABSTAIN_TOKEN}\n"
    "3. Every sentence MUST end with a citation to the passage id(s) it uses, like [abc123def456].\n"
    "4. Be concise and factual."
)


def format_context(chunks: list[RetrievedChunk]) -> str:
    return "\n\n".join(f"[{c.id}] (title: {c.title}) {c.text}" for c in chunks)

In [28]:
_CITE_RE = re.compile(r"\[([0-9a-f]{12})\]")


def parse_citations(text: str, valid_ids: set[str]) -> tuple[list[str], str]:
    found = _CITE_RE.findall(text)
    valid = [c for c in dict.fromkeys(found) if c in valid_ids]
    invalid = [c for c in dict.fromkeys(found) if c not in valid_ids]
    cleaned = text
    for bad in invalid:                       # strip hallucinated citation markers
        cleaned = cleaned.replace(f"[{bad}]", "")
    return valid, cleaned


@dataclass
class CitedAnswer:
    text: str
    cited_ids: list[str]
    abstained: bool
    raw: str


class CitedGenerator:
    def __init__(self, llm: LocalLLM):
        self.llm = llm

    def generate(self, question: str, chunks: list[RetrievedChunk]) -> CitedAnswer:
        ctx = format_context(chunks)
        user = f"Context passages:\n{ctx}\n\nQuestion: {question}\n\nAnswer:"
        raw = self.llm.chat(GENERATION_SYSTEM_PROMPT, user, max_tokens=400).strip()
        if ABSTAIN_TOKEN in raw:
            return CitedAnswer(text="", cited_ids=[], abstained=True, raw=raw)
        valid_ids = {c.id for c in chunks}
        cited, cleaned = parse_citations(raw, valid_ids)
        return CitedAnswer(text=cleaned.strip(), cited_ids=cited, abstained=False, raw=raw)


generator = CitedGenerator(llm)

if llm.is_up():
    _ans = generator.generate(_q.question, _reranked)
    print(f"Q: {_q.question}")
    print(f"abstained={_ans.abstained}  citations={_ans.cited_ids}")
    print(f"A: {_ans.text[:400]}")
else:
    print("[gen] model not loaded; skipping live demo")

Q: Were Scott Derrickson and Ed Wood of the same nationality?
abstained=True  citations=[]
A: 


## 10 · Atomic-claim verification gate (+ Chain-of-Verification)

**Theory.** **Hallucination firewall #2** — the decisive one. We split the drafted answer
into **atomic claims**, then check each claim against its cited context with the
faithfulness NLI verifier. A claim below `TAU_CLAIM` is *unsupported*; the gate fails and
the answer is downgraded to an abstention. For borderline answers, **Chain-of-Verification**
(CoVe) asks the model to re-derive each claim from context and revise. The point: an answer
reaches the user only if *every* claim is grounded.

**Learning note.** We use a small NLI cross-encoder here instead of the 7B MiniCheck model
or an LLM judge, so the gate is fast enough for a laptop. The principle is identical.


In [29]:
CLAIM_DECOMP_PROMPT = (
    "Split the answer into a numbered list of atomic, independently checkable factual "
    "claims. Drop opinions and citation markers. One claim per line.\nAnswer: {a}"
)


class ClaimExtractor:
    def __init__(self, llm: LocalLLM):
        self.llm = llm

    def extract(self, answer: str) -> list[str]:
        clean = _CITE_RE.sub("", answer).strip()
        if not clean:
            return []
        out = self.llm.chat("You extract atomic factual claims.",
                            CLAIM_DECOMP_PROMPT.format(a=clean), max_tokens=300)
        claims = [re.sub(r"^\s*\d+[.)]\s*", "", ln).strip(" -•\t")
                  for ln in out.splitlines() if ln.strip()]
        return [c for c in claims if len(c) > 3]

In [30]:
@dataclass
class ClaimVerdict:
    claim: str
    score: float
    supported: bool
    nli: float
    minicheck: float | None


@dataclass
class GateResult:
    passed: bool
    verdicts: list[ClaimVerdict]
    min_support: float
    n_claims: int


class VerificationGate:
    def __init__(self, verifier: Verifier, extractor: ClaimExtractor, tau_claim: float):
        self.verifier, self.extractor, self.tau = verifier, extractor, tau_claim

    def check(self, cited: CitedAnswer, chunks: list[RetrievedChunk]) -> GateResult:
        claims = self.extractor.extract(cited.text)
        used = [c for c in chunks if c.id in set(cited.cited_ids)] or chunks
        context = "\n\n".join(c.text for c in used)   # the LLM judge handles long multi-passage context
        verdicts = []
        for cl in claims:
            s = self.verifier.support(cl, context)
            verdicts.append(ClaimVerdict(cl, s["score"], s["score"] >= self.tau,
                                         s["nli"], s["minicheck"]))
        min_support = min((v.score for v in verdicts), default=0.0)
        passed = len(verdicts) > 0 and all(v.supported for v in verdicts)
        return GateResult(passed, verdicts, min_support, len(verdicts))

In [31]:
COVE_PROMPT = (
    "Revise the answer so EVERY sentence is directly supported by the context. Remove or "
    "soften any claim not supported. Keep citations [id].\n\nContext:\n{ctx}\n\n"
    "Answer:\n{ans}\n\nRevised answer:"
)


def cove_revise(answer: str, chunks: list[RetrievedChunk], llm: LocalLLM) -> str:
    ctx = format_context(chunks)
    return llm.chat("You make answers strictly faithful to context.",
                    COVE_PROMPT.format(ctx=ctx, ans=answer), max_tokens=400).strip()


claim_extractor = ClaimExtractor(llm)
gate = VerificationGate(verifier, claim_extractor, TAU_CLAIM)

if llm.is_up() and not _ans.abstained:
    _gr = gate.check(_ans, _reranked)
    print(f"claims={_gr.n_claims}  passed={_gr.passed}  min_support={_gr.min_support:.2f}")
    for v in _gr.verdicts:
        print(f"   [{'OK ' if v.supported else 'XX '}{v.score:.2f}] {v.claim[:80]}")
else:
    print("[gate] skipping live demo (model not loaded or model abstained)")

[gate] skipping live demo (model not loaded or model abstained)


## 11 · Abstention policy & structured output

**Theory.** Abstention is a *correct* answer, not a failure. We combine four signals into
one decision against `TAU_ABSTAIN`: (a) router said `no_retrieval`, (b) false-premise
detected, (c) the generator emitted the abstain token, (d) the verification gate failed
(a claim's support < `TAU_CLAIM`). Optionally we add **semantic entropy** — sampling
several answers and measuring disagreement — as an uncertainty signal. The output is a
strict, auditable schema so evaluation can parse answer-vs-abstain deterministically.

In [32]:
def semantic_entropy(question: str, gen_fn, n_samples: int = 5) -> float:
    """Cheap uncertainty proxy: sample answers, measure cluster entropy of their wording."""
    samples = []
    for _ in range(n_samples):
        try:
            samples.append(gen_fn(question))
        except Exception:
            pass
    if not samples:
        return 0.0
    from collections import Counter
    norm = [re.sub(r"\W+", " ", s.lower()).strip()[:80] for s in samples]
    counts = Counter(norm)
    probs = np.array(list(counts.values())) / len(norm)
    return float(-(probs * np.log(probs + 1e-12)).sum())


@dataclass
class FinalAnswer:
    status: str               # "answered" | "abstained"
    answer: str
    citations: list[str]
    min_support: float
    reason: str
    trace: dict = field(default_factory=dict)

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


class AbstentionPolicy:
    def __init__(self, tau_abstain: float):
        self.tau = tau_abstain

    def decide(self, route: str, false_premise: bool, cited: CitedAnswer,
               gate: GateResult | None) -> FinalAnswer:
        if route == "no_retrieval":
            return self._abstain("routed_no_retrieval", gate)
        if cited.abstained:
            return self._abstain("model_abstained", gate)
        if gate is None or not gate.passed or gate.min_support < self.tau:
            return self._abstain("unsupported_claims", gate)
        return FinalAnswer("answered", cited.text, cited.cited_ids,
                           gate.min_support, "verified", {})

    def _abstain(self, reason: str, gate: GateResult | None) -> FinalAnswer:
        ms = gate.min_support if gate else 0.0
        msg = ("I don't have enough supporting evidence in the available sources to answer "
               "this confidently.")
        return FinalAnswer("abstained", msg, [], ms, reason, {})


abstention_policy = AbstentionPolicy(TAU_ABSTAIN)
print("AbstentionPolicy ready; reasons = "
      "{routed_no_retrieval, false_premise, model_abstained, unsupported_claims, verified}")

AbstentionPolicy ready; reasons = {routed_no_retrieval, false_premise, model_abstained, unsupported_claims, verified}


## 12 · The agent — LangGraph CRAG / Self-RAG loop (capstone)

**Theory.** Now we wire the components into a stateful graph that *corrects itself*.
`route → retrieve → grade (CRAG)`. If evidence is strong, generate; if weak, **refine and
re-retrieve** (up to `MAX_HOPS`); if hopeless, abstain without generating (the #1 cause of
hallucination is generating from bad context). After generation we **verify**, then
**finalize** through the abstention policy. The loop is bounded so latency stays in budget.

In [33]:
from typing import TypedDict, Any


class AgentState(TypedDict, total=False):
    question: str
    route: str
    false_premise: bool
    query: str
    evidence: list
    grade: float
    draft: Any
    gate: Any
    final: Any
    hops: int
    trace: list
    latencies: dict


GRADE_PROMPT = (
    "On a scale 0.0-1.0, how well do these passages let you fully answer the question? "
    "Reply with only a number.\nQuestion: {q}\nPassages:\n{ctx}"
)


def grade_evidence(query: str, chunks: list[RetrievedChunk], llm: LocalLLM) -> float:
    ctx = "\n".join(f"- {c.text[:200]}" for c in chunks[:8])
    out = llm.chat("You grade retrieval sufficiency.",
                   GRADE_PROMPT.format(q=query, ctx=ctx), max_tokens=8)
    m = re.search(r"[01](?:\.\d+)?", out)
    return float(m.group()) if m else 0.5

In [34]:
def _timed(state: AgentState, stage: str, dt: float):
    state.setdefault("latencies", {})[stage] = state.get("latencies", {}).get(stage, 0.0) + dt
    state.setdefault("trace", []).append((stage, round(dt, 3)))


def n_route(state: AgentState) -> AgentState:
    t = time.time()
    state["route"] = router.route(state["question"])
    state["false_premise"] = detect_false_premise(state["question"], llm)
    state["query"] = state["question"]
    state["hops"] = 0
    _timed(state, "route", time.time() - t)
    return state


def n_retrieve(state: AgentState) -> AgentState:
    t = time.time()
    state["evidence"] = retrieve_and_rerank(state["query"])
    _timed(state, "retrieve", time.time() - t)
    return state


def n_grade(state: AgentState) -> AgentState:
    t = time.time()
    state["grade"] = grade_evidence(state["question"], state.get("evidence", []), llm)
    _timed(state, "grade", time.time() - t)
    return state


def n_refine(state: AgentState) -> AgentState:
    t = time.time()
    state["hops"] = state.get("hops", 0) + 1
    subs = decomposer.decompose(state["question"])
    state["query"] = " ".join(subs)        # broaden the query with sub-questions
    _timed(state, "refine", time.time() - t)
    return state


def n_generate(state: AgentState) -> AgentState:
    t = time.time()
    state["draft"] = generator.generate(state["question"], state.get("evidence", []))
    _timed(state, "generate", time.time() - t)
    return state


def n_verify(state: AgentState) -> AgentState:
    t = time.time()
    draft = state["draft"]
    state["gate"] = None if draft.abstained else gate.check(draft, state["evidence"])
    _timed(state, "verify", time.time() - t)
    return state


def n_finalize(state: AgentState) -> AgentState:
    draft = state.get("draft") or CitedAnswer("", [], True, "")
    state["final"] = abstention_policy.decide(
        state.get("route", "single_hop"), state.get("false_premise", False), draft, state.get("gate"))
    state["final"].trace = {"route": state.get("route"), "hops": state.get("hops", 0),
                            "grade": state.get("grade"), "latencies": state.get("latencies", {})}
    return state

In [35]:
from langgraph.graph import StateGraph, END


def _after_route(state: AgentState) -> str:
    # false-premise is recorded as a signal but is NOT a hard abstain — an 8B yes/no
    # detector is too noisy; the evidence path (grade + claim verification) decides.
    return "finalize" if state["route"] == "no_retrieval" else "retrieve"


def _after_grade(state: AgentState) -> str:
    g = state.get("grade", 0.0)
    if g >= CRAG_OK:
        return "generate"
    if g < CRAG_BAD or state.get("hops", 0) >= MAX_HOPS:
        return "generate" if g >= CRAG_BAD else "finalize"   # too weak -> abstain
    return "refine"


def build_agent_graph():
    g = StateGraph(AgentState)
    for name, fn in [("route", n_route), ("retrieve", n_retrieve), ("grade", n_grade),
                     ("refine", n_refine), ("generate", n_generate), ("verify", n_verify),
                     ("finalize", n_finalize)]:
        g.add_node(name, fn)
    g.set_entry_point("route")
    g.add_conditional_edges("route", _after_route, {"retrieve": "retrieve", "finalize": "finalize"})
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", _after_grade,
                            {"generate": "generate", "refine": "refine", "finalize": "finalize"})
    g.add_edge("refine", "retrieve")
    g.add_edge("generate", "verify")
    g.add_edge("verify", "finalize")
    g.add_edge("finalize", END)
    return g.compile()


@dataclass
class RAGResult:
    question: str
    final: FinalAnswer
    route: str
    hops: int
    grade: float
    latencies: dict
    draft_text: str = ""          # the answer BEFORE abstention (for the risk-coverage sweep)
    min_support: float | None = None
    hard_abstain: bool = False    # abstained for a non-gate reason (route/false-premise/model)


class RAGAgent:
    def __init__(self):
        self.app = build_agent_graph()

    def answer(self, question: str) -> RAGResult:
        t = time.time()
        st = self.app.invoke({"question": question})
        st.setdefault("latencies", {})["total"] = time.time() - t
        draft = st.get("draft")
        gate_r = st.get("gate")
        hard = st["final"].reason in {"routed_no_retrieval", "false_premise", "model_abstained"}
        return RAGResult(question, st["final"], st.get("route", ""), st.get("hops", 0),
                         st.get("grade", 0.0), st["latencies"],
                         draft_text=(draft.text if draft else ""),
                         min_support=(gate_r.min_support if gate_r else None),
                         hard_abstain=hard)


if llm.is_up():
    agent = RAGAgent()
    _r = agent.answer(_q.question)
    print(f"Q: {_q.question}")
    print(f"route={_r.route} hops={_r.hops} grade={_r.grade:.2f} status={_r.final.status} "
          f"reason={_r.final.reason}")
    print(f"A: {_r.final.answer[:300]}")
    print(f"latencies(s): { {k: round(v,2) for k,v in _r.latencies.items()} }")
else:
    agent = None
    print("[agent] model not loaded; agent unavailable")

Q: Were Scott Derrickson and Ed Wood of the same nationality?
route=no_retrieval hops=0 grade=0.00 status=abstained reason=routed_no_retrieval
A: I don't have enough supporting evidence in the available sources to answer this confidently.
latencies(s): {'route': 2.6, 'total': 2.62}


## 13 · Evaluation: golden set + 2×2 confusion + risk–coverage curve

**Theory.** The headline scorecard. We assemble a golden set with **answerable**
(HotpotQA) and **unanswerable** (SQuAD-v2 `is_impossible` + false-premise) strata, run the
agent over all of it, and score it as a **2×2**: {answerable, unanswerable} × {answered,
abstained}. The dangerous cell is *unanswerable × answered* = hallucination. Then we sweep
the abstention threshold τ to draw the **risk–coverage curve** (hallucination-rate vs
coverage) and pick the τ that holds hallucination under a budget while keeping the most
coverage. This is how we *prove* "near-zero on the answerable subset."

In [36]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def savefig(name: str) -> str:
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[fig] {p}")
    return str(p)


@dataclass
class EvalItem:
    qid: str
    question: str
    gold_answer: str
    gold_titles: list[str]
    answerable: bool
    source: str


def build_false_premise_set() -> list[EvalItem]:
    qs = [
        "In what year did Albert Einstein win his second Nobel Prize in Physics?",
        "Which Academy Award did the novel Pride and Prejudice win in 1814?",
        "What was the name of the spaceship Marie Curie flew to the Moon?",
        "How many gold medals did William Shakespeare win at the Olympics?",
        "Which programming language did Isaac Newton invent in 1700?",
        "What is the population of the underwater city founded by Nikola Tesla?",
    ]
    return [EvalItem(f"fp_{i}", q, "", [], False, "false_premise") for i, q in enumerate(qs)]


def build_golden_set(qa_items, n_ans: int, n_unans: int) -> list[EvalItem]:
    rng = random.Random(SEED)
    ans = rng.sample(qa_items, min(n_ans, len(qa_items)))
    items = [EvalItem(q.qid, q.question, q.answer, q.gold_titles, True, "hotpotqa") for q in ans]
    # unanswerable: SQuAD v2 impossible
    try:
        sq = load_squad_v2()
        imp = [r for r in sq if r["answers"]["text"] == [] or len(r["answers"]["text"]) == 0]
        for r in rng.sample(imp, min(max(0, n_unans - 6), len(imp))):
            items.append(EvalItem(r["id"], r["question"], "", [], False, "squad_v2"))
    except Exception as e:
        print(f"[golden] squad_v2 skipped ({e})")
    items += build_false_premise_set()
    rng.shuffle(items)
    return items


golden = build_golden_set(qa_items, N_EVAL_ANSWERABLE, N_EVAL_UNANSWERABLE)
print(f"[golden] {len(golden)} items  "
      f"(answerable={sum(i.answerable for i in golden)}, "
      f"unanswerable={sum(not i.answerable for i in golden)})")

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

[golden] 40 items  (answerable=20, unanswerable=20)


In [37]:
_ARTICLES = {"a", "an", "the"}


def normalize_ans(s: str) -> str:
    s = re.sub(r"[^a-z0-9 ]", " ", s.lower())
    return " ".join(w for w in s.split() if w not in _ARTICLES)


def token_f1(pred: str, gold: str) -> float:
    p, g = normalize_ans(pred).split(), normalize_ans(gold).split()
    if not p or not g:
        return 0.0
    common = 0
    gc = list(g)
    for w in p:
        if w in gc:
            gc.remove(w); common += 1
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)


def is_correct(pred: str, gold: str) -> bool:
    if not gold:
        return False
    return normalize_ans(gold) in normalize_ans(pred) or token_f1(pred, gold) >= 0.5

In [38]:
def run_pipeline_over_set(agent: "RAGAgent", items: list[EvalItem]) -> list[RAGResult]:
    results = []
    for k, it in enumerate(items):
        try:
            results.append(agent.answer(it.question))
        except Exception as e:
            print(f"   ! item {k} failed: {e}")
            results.append(RAGResult(it.question, FinalAnswer("abstained", "", [], 0.0, "error"),
                                     "", 0, 0.0, {"total": 0.0}, hard_abstain=True))
        if (k + 1) % max(1, len(items) // 10) == 0:
            print(f"   ...{k+1}/{len(items)}")
    return results

In [39]:
def confusion_2x2(results: list[RAGResult], items: list[EvalItem]) -> np.ndarray:
    cm = np.zeros((2, 2), dtype=int)   # rows: ans/unans ; cols: answered/abstained
    for r, it in zip(results, items):
        i = 0 if it.answerable else 1
        j = 0 if r.final.status == "answered" else 1
        cm[i, j] += 1
    return cm


def plot_confusion(cm: np.ndarray) -> str:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["answered", "abstained"])
    ax.set_yticks([0, 1], ["answerable", "unanswerable"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16)
    ax.set_title("2x2: hallucinations live in (unanswerable, answered)")
    return savefig("confusion_2x2")


def risk_coverage_curve(results, items, taus) -> "pd.DataFrame":
    rows = []
    for tau in taus:
        answered = halluc = 0
        for r, it in zip(results, items):
            decide_answer = (not r.hard_abstain and r.min_support is not None
                             and r.min_support >= tau)
            if decide_answer:
                answered += 1
                wrong = (not it.answerable) or (not is_correct(r.draft_text, it.gold_answer))
                halluc += int(wrong)
        rows.append({"tau": tau, "coverage": answered / len(items),
                     "hallucination_rate": (halluc / answered) if answered else 0.0,
                     "answered": answered})
    return pd.DataFrame(rows)


def pick_tau(df: "pd.DataFrame", max_halluc: float = 0.05) -> float:
    ok = df[df["hallucination_rate"] <= max_halluc]
    return float(ok.sort_values("coverage", ascending=False).iloc[0]["tau"]) if len(ok) else 1.0


def plot_risk_coverage(df: "pd.DataFrame", tau_star: float) -> str:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.plot(df["coverage"], df["hallucination_rate"], "-o", ms=3)
    row = df.iloc[(df["tau"] - tau_star).abs().argmin()]
    ax.scatter([row["coverage"]], [row["hallucination_rate"]], color="red", zorder=5,
               label=f"τ*={tau_star:.2f}")
    ax.set_xlabel("coverage (fraction answered)")
    ax.set_ylabel("hallucination rate (of answered)")
    ax.set_title("Risk–coverage: the price of near-zero hallucination")
    ax.legend()
    return savefig("risk_coverage")

In [40]:
def compute_metrics(results, items) -> dict:
    answered = [(r, it) for r, it in zip(results, items) if r.final.status == "answered"]
    faith = float(np.mean([r.min_support for r, _ in answered if r.min_support is not None])) \
        if answered else float("nan")
    # answer relevancy = cosine(answer, question) via the online embedder
    rel = float("nan")
    if answered:
        qs = [it.question for _, it in answered]
        ans = [r.final.answer for r, _ in answered]
        qv = embed_texts(online_embedder, qs, is_query=True)
        av = embed_texts(online_embedder, ans)
        rel = float(np.mean([float(np.dot(a, b)) for a, b in zip(qv, av)]))
    # answerable accuracy + retrieval recall (re-retrieve for the answerable golden items)
    ans_items = [it for it in items if it.answerable]
    recall = float(np.mean([passage_recall(retrieve_and_rerank(it.question), it.gold_titles)
                            for it in ans_items[:50]])) if ans_items else float("nan")
    correct = [is_correct(r.draft_text, it.gold_answer)
               for r, it in zip(results, items) if it.answerable]
    acc = float(np.mean(correct)) if correct else float("nan")
    return {"faithfulness": round(faith, 3), "answer_relevancy": round(rel, 3),
            "context_recall@k": round(recall, 3), "answerable_accuracy": round(acc, 3)}

In [41]:
if agent is not None:
    EVAL_RESULTS = run_pipeline_over_set(agent, golden)
    CM = confusion_2x2(EVAL_RESULTS, golden)
    TAUS = np.round(np.linspace(0.0, 1.0, 21), 2)
    RC = risk_coverage_curve(EVAL_RESULTS, golden, TAUS)
    TAU_STAR = pick_tau(RC, max_halluc=0.05)
    METRICS = compute_metrics(EVAL_RESULTS, golden)
    plot_confusion(CM)
    plot_risk_coverage(RC, TAU_STAR)
    print("\nconfusion (rows ans/unans, cols answered/abstained):\n", CM)
    print(f"\nchosen τ* (halluc<=5%): {TAU_STAR}")
    print("metrics:", json.dumps(METRICS, indent=2))
    _unans_answered = int(CM[1, 0])
    print(f"hallucinations (unanswerable answered): {_unans_answered} / {CM[1].sum()} unanswerable")
else:
    EVAL_RESULTS, CM, RC, TAU_STAR, METRICS = [], None, None, None, {}
    print("[eval] agent unavailable")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

   ...4/40
   ...8/40


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[fig] /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/out/figures/confusion_2x2.png
[fig] /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/out/figures/risk_coverage.png

confusion (rows ans/unans, cols answered/abstained):
 [[ 0 20]
 [ 0 20]]

chosen τ* (halluc<=5%): 0.0
metrics: {
  "faithfulness": NaN,
  "answer_relevancy": NaN,
  "context_recall@k": 0.825,
  "answerable_accuracy": 0.15
}
hallucinations (unanswerable answered): 0 / 20 unanswerable


## 14 · Validating the verifier itself (HaluBench)

**Theory.** The faithfulness verifier is a model too — we must not trust it blind. We run
it over **HaluBench** (human-labeled faithful/hallucinated answers) and report **AUROC**
and the ROC curve, then calibrate `TAU_CLAIM` from the operating point. An unvalidated
verifier just moves the hallucination from the answer into the scorecard.

In [42]:
# ---- validate the NLI verifier on HaluBench (if available) ------------------
def eval_verifier(verifier: Verifier, n: int = 300) -> dict:
    hb = load_halubench()
    if hb is None:
        return {}
    hb = hb.shuffle(seed=SEED).select(range(min(n, len(hb))))   # mix PASS/FAIL (dataset is grouped)
    scores, labels = [], []
    for ex in hb:
        passage = ex.get("passage") or ex.get("context") or ""
        answer = ex.get("answer") or ex.get("response") or ""
        lab = str(ex.get("label", "")).upper()
        if not passage or not answer:
            continue
        scores.append(verifier.nli_score(answer, passage))
        labels.append(1 if lab.startswith("PASS") else 0)   # PASS = faithful (positive)
    from sklearn.metrics import roc_auc_score, roc_curve
    auroc = float(roc_auc_score(labels, scores)) if len(set(labels)) > 1 else float("nan")
    fpr, tpr, thr = roc_curve(labels, scores) if len(set(labels)) > 1 else ([], [], [])
    return {"auroc": round(auroc, 3), "n": len(labels), "fpr": list(fpr),
            "tpr": list(tpr), "thr": list(thr)}


def plot_roc(roc: dict) -> str:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.plot(roc["fpr"], roc["tpr"], "-", label=f"NLI AUROC={roc['auroc']}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
    ax.set_title("Verifier ROC on HaluBench"); ax.legend()
    return savefig("verifier_roc")


if agent is not None:
    VERIFIER_EVAL = eval_verifier(verifier, n=min(100, len(golden) * 2))
    if VERIFIER_EVAL:
        plot_roc(VERIFIER_EVAL)
        print(f"[verifier] AUROC={VERIFIER_EVAL['auroc']} over n={VERIFIER_EVAL['n']} HaluBench items")
else:
    VERIFIER_EVAL = {}
    print("[verifier-eval] agent not available")


README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.51MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/14900 [00:00<?, ? examples/s]

[fig] /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/out/figures/verifier_roc.png
[verifier] AUROC=0.407 over n=80 HaluBench items


## 15 · Scale lab — skipped in learning mode

**Theory.** The full notebook literally builds LanceDB indexes at 100k / 1M / 10M vectors
on an H100-class machine to prove the scale claim. That requires ~80GB VRAM, fast NVMe, and
many minutes of indexing time — far beyond a laptop.

**Learning note.** `RUN_SCALE_LAB=False` in this notebook, so this section only prints a
note. The hallucination-control logic you have already seen does not change at scale; only
the indexing and ANN parameters do.


In [44]:
# ---- scale lab placeholder (keeps downstream references valid) ---------------
SCALE_DF, SCALE_EXTRAP, SCALE_COST = None, {}, {}
if RUN_SCALE_LAB:
    print("[scale] RUN_SCALE_LAB=True is not supported in the learning edition; skipping.")
else:
    print("[scale] skipped (RUN_SCALE_LAB=False). Re-run on the H100 environment for real scale metrics.")


[scale] skipped (RUN_SCALE_LAB=False). Re-run on the H100 environment for real scale metrics.


## 16 · Latency & cost dashboard

**Theory.** Per-stage latency attribution tells us where the agentic budget goes
(generation and verification dominate). Cost = wall-clock × $1.90/hr plus amortized
offline indexing. We consolidate everything into one headline-metrics card.

In [45]:
def aggregate_latencies(results: list[RAGResult]) -> "pd.DataFrame":
    stages: dict[str, list] = {}
    for r in results:
        for k, v in r.latencies.items():
            stages.setdefault(k, []).append(v)
    rows = [{"stage": k, "p50_s": round(float(np.percentile(v, 50)), 3),
             "p95_s": round(float(np.percentile(v, 95)), 3),
             "mean_s": round(float(np.mean(v)), 3)} for k, v in stages.items()]
    return pd.DataFrame(rows).sort_values("mean_s", ascending=False)


def plot_stage_latency(df: "pd.DataFrame") -> str:
    d = df[df["stage"] != "total"]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.barh(d["stage"], d["p95_s"])
    ax.set_xlabel("p95 latency (s)"); ax.set_title("Per-stage latency (p95)")
    ax.invert_yaxis()
    return savefig("stage_latency")


def headline_metrics(metrics, cm, tau_star, verifier_eval, scale_cost, lat_df) -> dict:
    out = dict(metrics)
    if cm is not None:
        unans = int(cm[1].sum())
        out["abstention_on_unanswerable"] = round(int(cm[1, 1]) / unans, 3) if unans else None
        out["hallucination_on_unanswerable"] = round(int(cm[1, 0]) / unans, 3) if unans else None
        ans = int(cm[0].sum())
        out["coverage_on_answerable"] = round(int(cm[0, 0]) / ans, 3) if ans else None
    out["tau_star"] = tau_star
    out["verifier_auroc"] = verifier_eval.get("auroc")
    out["scale_projection_100M"] = scale_cost
    if lat_df is not None and len(lat_df):
        tot = lat_df[lat_df["stage"] == "total"]
        out["latency_p95_s"] = float(tot["p95_s"].iloc[0]) if len(tot) else None
    return out


if agent is not None and EVAL_RESULTS:
    LAT_DF = aggregate_latencies(EVAL_RESULTS)
    plot_stage_latency(LAT_DF)
    print(LAT_DF.to_string(index=False))
    HEADLINE = headline_metrics(METRICS, CM, TAU_STAR, VERIFIER_EVAL, SCALE_COST, LAT_DF)
    print("\nHEADLINE:\n", json.dumps(HEADLINE, indent=2))
else:
    LAT_DF, HEADLINE = None, {}
    print("[dashboard] no evaluation results to display")

[fig] /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/out/figures/stage_latency.png
   stage  p50_s  p95_s  mean_s
generate  6.050 10.654   5.994
   total  1.228 22.880   5.678
  verify  1.855 18.708   5.301
  refine  4.978  5.635   4.853
   grade  4.456  7.246   4.258
retrieve  1.703  4.605   2.108
   route  1.093  4.789   1.661

HEADLINE:
 {
  "faithfulness": NaN,
  "answer_relevancy": NaN,
  "context_recall@k": 0.825,
  "answerable_accuracy": 0.15,
  "abstention_on_unanswerable": 1.0,
  "hallucination_on_unanswerable": 0.0,
  "coverage_on_answerable": 0.0,
  "tau_star": 0.0,
  "verifier_auroc": 0.407,
  "scale_projection_100M": {},
  "latency_p95_s": 22.88
}


## 17 · Conclusions & run manifest

**Takeaways.**
- *Near-zero hallucination = retrieve → constrain → verify → abstain.* The 2×2 and the
  risk–coverage curve quantify the trade: pushing hallucination toward zero costs some
  coverage, and τ* is a conscious choice, not an accident.
- *The learning edition uses the same firewall structure as the full pipeline,* just with
  small, laptop-friendly models. Swapping `GEN_MODEL`, `EMBED_MODEL`, `RERANK_MODEL`, and
  `NLI_MODEL` back to the larger names, increasing `SLICE_SIZE`, and setting
  `RUN_SCALE_LAB=True` reproduces the full H100 experiment.
- *Trust the verifier only after validating it* (HaluBench AUROC).

The manifest below captures config + metrics + artifact paths so the run is fully
reproducible and auditable.


In [46]:
def dump_run_manifest() -> dict:
    manifest = {
        "config": CFG.summary(),
        "gpu": GPU,
        "corpus": {"passages": len(corpus), "chunks": len(chunks)},
        "golden": {"n": len(golden),
                   "answerable": sum(i.answerable for i in golden),
                   "unanswerable": sum(not i.answerable for i in golden)},
        "headline": HEADLINE,
        "scale_measured": (SCALE_DF.to_dict("records") if SCALE_DF is not None else []),
        "scale_projection_100M": SCALE_COST,
        "figures": [str(p) for p in sorted(FIG_DIR.glob("*.png"))],
    }
    path = OUT_DIR / "run_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"[manifest] {path}")
    return manifest


MANIFEST = dump_run_manifest()
print(json.dumps(MANIFEST.get("headline", {}), indent=2))
print("\nDone. Pull artifacts with scripts/pull_results.sh")

[manifest] /Users/souravchaurasia/Desktop/Personal/rag-at-scale/notebooks/learning_data/out/run_manifest.json
{
  "faithfulness": NaN,
  "answer_relevancy": NaN,
  "context_recall@k": 0.825,
  "answerable_accuracy": 0.15,
  "abstention_on_unanswerable": 1.0,
  "hallucination_on_unanswerable": 0.0,
  "coverage_on_answerable": 0.0,
  "tau_star": 0.0,
  "verifier_auroc": 0.407,
  "scale_projection_100M": {},
  "latency_p95_s": 22.88
}

Done. Pull artifacts with scripts/pull_results.sh
